# Tessera on MNIST — symbolic features + linear classifier

**Goal (per `docs/PROJECT_GOALS.md`):** ≥ 95% accuracy on full MNIST 10-class with the JAX backend.

**Approach:** symbolic regression discovers K class-specific features (one-vs-rest per digit), then a linear classifier (logistic regression) is trained on the stacked features.

```
image (28×28)
   │
   ├─→ SR feature 1: f₁(img) → scalar  (one-vs-rest target for digit 0)
   ├─→ SR feature 2: f₂(img) → scalar  (one-vs-rest target for digit 1)
   │   ...
   └─→ SR feature K: fₖ(img) → scalar  (one-vs-rest target for digit K-1)
         │
         ▼
      [N × K] feature matrix
         │
         ▼
    LogisticRegression → 10-class predictions
```

**Honest expectations:**
- Single-feature SR gets ~80-85% on 0-vs-rest binary. With K=10 features one-vs-rest, the linear ensemble should reach ~85-92% on 10-class.
- Reaching 95% likely needs more features (K=20-50) and/or larger search budget. This notebook is **step 4** of the GPU roadmap — measure the ceiling first; the step-5 multi-feature ensemble is for if that ceiling is below 95%.
- **Time budget:** ~5-10 min per SR feature on Colab GPU with the configuration here. K=10 features = ~60-100 min total. Use a smaller subset if you want a faster cycle.

**What this notebook does NOT yet do:**
- Vmap the tree over the sample axis (per-image Python loop is still the bottleneck for the *search*, not the feature evaluation). Future work.
- `jax.grad`-based constant optimization (still scipy Nelder-Mead). Future work.
- The existing tessera.search.GP class assumes a single-env-dict; this notebook uses a CUSTOM per-image GP loop (same pattern as `benchmarks/run_mnist_feature_discovery.py`) because the tree evaluates ONE image at a time.

## Setup

In [ ]:
!nvidia-smi || echo 'No GPU detected — switch runtime to GPU before proceeding'

In [ ]:
!pip install --force-reinstall --no-deps -q git+https://github.com/davechendatascience/tessera.git
!pip install -q --upgrade "jax[cuda12]" scikit-learn

# After this cell: restart the Colab runtime (Runtime → Restart session)
# so Python's module cache picks up the fresh tessera. Then re-run
# from this cell forward.

In [ ]:
import time
import random
import numpy as np
import jax, jax.numpy as jnp

from tessera.expression import (
    Var, Const, complexity, used_features, FunctionalOp2D, evaluate,
    compile_image_predictor, clear_image_predictor_cache,
)
from tessera.expression.simplify import simplify_canonical
from tessera.expression.mutation import random_tree, mutate, validate_tree

print('JAX version:', jax.__version__)
print('Devices:', jax.devices())

## Load MNIST

In [ ]:
from sklearn.datasets import fetch_openml

print('Loading MNIST via sklearn (cached after first run)...')
t0 = time.time()
mnist = fetch_openml('mnist_784', version=1, as_frame=False, cache=True, parser='auto')
X_all = mnist.data.reshape(-1, 28, 28).astype(np.float32) / 255.0
y_all = mnist.target.astype(int)
print(f'Loaded {X_all.shape[0]} samples, shape {X_all.shape[1:]} in {time.time()-t0:.1f}s')

# Subset for tractability. The full 60K is too slow with per-image loop.
# Start with 5K train / 1K test; scale up after validating the pipeline.
N_TRAIN = 5000
N_TEST = 1000
IMG_SIZE = 14    # downsample 28×28 → 14×14 by 2× block mean for further speed

rng = np.random.default_rng(2026)
perm = rng.permutation(X_all.shape[0])
train_idx, test_idx = perm[:N_TRAIN], perm[N_TRAIN:N_TRAIN+N_TEST]
X_train_full, y_train = X_all[train_idx], y_all[train_idx]
X_test_full, y_test = X_all[test_idx], y_all[test_idx]

def downsample(a):
    return a.reshape(-1, IMG_SIZE, 2, IMG_SIZE, 2).mean(axis=(2, 4)).astype(np.float64)

X_train = downsample(X_train_full)
X_test = downsample(X_test_full)
print(f'Train: {X_train.shape}, Test: {X_test.shape}')
print(f'Class balance (train): {np.bincount(y_train) / len(y_train)}')

## Single-image tree evaluation

The tree expects a 2D field. We evaluate it on each image, take the mean of the result as the scalar prediction. This is the standard CNN-feature-then-pool pattern, except the convolution kernel is *discovered* by SR.

In [ ]:
# Tier-3 batched predictor: vmap + jit a tree over the image batch axis.
# First call per tree compiles (~100-200ms); subsequent calls are ~1ms
# regardless of N. ~30x faster than the per-image Python loop in our
# local benchmark (N=1000, 14x14 images, Laplacian feature).
#
# Module-level cache: same tree compiled once, reused across all
# scoring calls within a feature run.

def predict_batch(tree, X):
    """Apply tree to each image in X via jit+vmap. Returns numpy [N]."""
    X_jax = jnp.asarray(X)
    try:
        fn = compile_image_predictor(tree, batch_var='image', reduce='mean')
    except Exception as e:
        # Some trees fail to jit-compile (NaN-rich, shape mismatch, etc).
        # Return all-NaN; the scorer will reject via min_valid_frac.
        return np.full(len(X), float('nan'))
    try:
        out_jax = fn(X_jax)
        out = np.asarray(out_jax)
    except Exception:
        return np.full(len(X), float('nan'))
    return out


# Kept for back-compat with the K=1 smoke-test cell — single-image
# eval is fine to leave as a Python loop for ergonomic debug printing.
def evaluate_on_image(tree, image):
    try:
        out = evaluate(tree, {'image': image})
    except Exception:
        return float('nan')
    if np.isscalar(out):
        return float(out) if np.isfinite(out) else float('nan')
    out = np.asarray(out, dtype=np.float64)
    if out.ndim == 0:
        return float(out)
    finite_mask = np.isfinite(out)
    if not finite_mask.any():
        return float('nan')
    return float(out[finite_mask].mean())

### Per-image loop replaced with jit + vmap

The previous version of this notebook used a Python `for` loop over images, calling `evaluate(tree, {'image': img})` per sample. That made each `predict_batch` call O(N) Python overhead; the GP's scoring step (60 trees × 20 gens × N=1000 balanced samples) added up to **5-15 min per feature** in the user's first run.

The new `predict_batch` above uses `compile_image_predictor` (from the `tessera.expression.jit` module). For each tree it builds `jax.jit(jax.vmap(per_sample_fn))` — one XLA kernel evaluates the tree across the entire image batch in parallel. After a one-time compile (~100-200 ms per unique tree), each batch eval is ~1 ms regardless of N. Local benchmark: **31× speedup** at N=1000, 14×14 images.

Expected effect on the K=10 run: per-feature time should drop from ~5-15 min to ~30-90 s, making the full 10-feature ensemble tractable in ~10-20 min instead of ~60-100 min.

## Custom GP loop for one-vs-rest feature discovery

Targets binary `y = 1.0 if digit == k else 0.0`. The SR finds a feature whose mean-pooled output discriminates class k from others.

### Diagnostic update (2026-05-24)

**First K=10 run gave 71.1% test accuracy (vs 95% target).** Inspecting the discovered trees revealed two failure modes:

1. **Dead branches** — several features contained `M2D[kernel](Const)` or `V2[...](Const)`. A 2D measure applied to a constant scalar is ITSELF a constant (= `c · Σ kernel`), so those branches contributed nothing but inflated tree complexity. The GP wasted budget carrying them.

2. **Parsimony too low** — at `parsimony=0.001 × cx=15 = 0.015` added to losses in the 0.15-0.25 range, the dead branches barely cost anything. Bumped to `parsimony=0.01` so cx=15 contributes 0.15 — meaningful penalty.

Both fixes shipped:
- `simplify_canonical` now folds `FunctionalOp(Const)`, `FunctionalOp2D(Const)`, `Volterra2(Const)`, `SeparableBilinear(Const, Const)`, and `X/X → 1`.
- The default in `discover_feature_one_vs_rest` is now `parsimony=0.01`.

**Expected effect**: discovered features get cleaner (fewer dead branches, lower cx for same signal), per-feature loss drops 0.02-0.04, K=10 ensemble accuracy should rise ~3-6 pts to **~74-77%**. Still below 95% — the architectural gap (single-feature SR + linear classifier ≠ CNN) requires more features per class, multi-scale, or hybrid SR+MLP to close (see `docs/PROJECT_GOALS.md` Goal 3 escalation paths).

In [ ]:
def score_tree(tree, X, y, parsimony, cx):
    preds = predict_batch(tree, X)
    mask = np.isfinite(preds)
    if mask.sum() < len(preds) * 0.9:
        return float('inf')
    err = preds[mask] - y[mask]
    return float(np.mean(err ** 2)) + parsimony * cx


def _is_valid_image_tree(tree, feature_names):
    """Tree must (a) pass basic validation, (b) actually use `image`.
    Rejects pure-constant trees that learn the class prior instead of
    discovering a feature."""
    if validate_tree(tree, set(feature_names)) is not None:
        return False
    if 'image' not in used_features(tree):
        return False
    return True


def discover_feature_one_vs_rest(X_train, y_train, target_class,
                                   pop_size=60, n_gens=20,
                                   parsimony=0.01, seed=0,
                                   verbose=False):
    """Run SR to find a feature distinguishing target_class from others.

    Two fixes vs the naive setup:
    (1) Class-balanced sampling 1:1 (all positives + same number of
        negatives) — removes the trivial constant-predictor local
        minimum at the class prior (~0.10 for 10-class MNIST gives
        MSE≈0.09 which trivially beats most random trees on imbalanced
        data; balancing pushes the constant MSE to 0.25, easy to
        beat with real features).
    (2) Require the discovered tree to USE `image` — pure-constant
        and bare-variable trees rejected via `used_features(tree)`.
    """
    feature_names = ['image']
    rng = random.Random(seed)

    # (1) Class-balanced sampling
    pos_idx = np.where(y_train == target_class)[0]
    neg_idx = np.where(y_train != target_class)[0]
    rng_np = np.random.default_rng(seed)
    if len(neg_idx) < len(pos_idx):
        # rare edge case; sample with replacement
        neg_sample = rng_np.choice(neg_idx, size=len(pos_idx), replace=True)
    else:
        neg_sample = rng_np.choice(neg_idx, size=len(pos_idx), replace=False)
    balanced_idx = np.concatenate([pos_idx, neg_sample])
    rng_np.shuffle(balanced_idx)
    X_balanced = X_train[balanced_idx]
    y_balanced = (y_train[balanced_idx] == target_class).astype(np.float64)
    if verbose:
        print(f'  digit {target_class}: balanced to {len(pos_idx)} pos + '
              f'{len(pos_idx)} neg = {len(balanced_idx)} samples')

    # Initial population
    pop = []
    attempts = 0
    # Bumped attempt budget (was 10x) because require-image rejection
    # filters out more random trees.
    while len(pop) < pop_size and attempts < pop_size * 30:
        attempts += 1
        tree = random_tree(rng, feature_names, max_depth=4,
                            enable_2d=True, pointwise_only=False)
        if not _is_valid_image_tree(tree, feature_names):
            continue
        tree = simplify_canonical(tree)
        # simplify may collapse to a Const — re-check
        if not _is_valid_image_tree(tree, feature_names):
            continue
        cx = complexity(tree)
        loss = score_tree(tree, X_balanced, y_balanced, parsimony, cx)
        if np.isfinite(loss):
            pop.append((tree, loss, cx))
    if len(pop) < pop_size:
        raise RuntimeError(
            f'Could not init {pop_size} valid trees for digit '
            f'{target_class} (got {len(pop)} in {attempts} attempts). '
            f'The require-image filter may be too restrictive; consider '
            f'lowering pop_size or extending the attempt budget.'
        )
    pop.sort(key=lambda x: x[1])

    # GP generations
    for gen in range(n_gens):
        t0 = time.time()
        offspring = []
        attempts = 0
        while len(offspring) < pop_size and attempts < pop_size * 10:
            attempts += 1
            a = min(rng.sample(pop, 3), key=lambda x: x[1])[0]
            b = min(rng.sample(pop, 3), key=lambda x: x[1])[0]
            child = mutate([a, b], rng, feature_names,
                            pointwise_only=False, enable_2d=True)
            if child is None:
                continue
            child = simplify_canonical(child)
            if not _is_valid_image_tree(child, feature_names):
                continue
            cx = complexity(child)
            loss = score_tree(child, X_balanced, y_balanced, parsimony, cx)
            if np.isfinite(loss):
                offspring.append((child, loss, cx))

        combined = pop + offspring
        combined.sort(key=lambda x: x[1])
        pop = combined[:pop_size]

        if verbose and (gen % 5 == 0 or gen == n_gens - 1):
            best = pop[0]
            print(f'  digit {target_class} gen {gen:2d}: '
                  f'loss={best[1]:.4g} cx={best[2]} ({time.time()-t0:.1f}s)')

    return pop[0][0]   # best tree

### Why balanced sampling + require-image?

Earlier runs found that several SR features converged to a **trivial constant predictor** at the class prior probability (~0.10 for one-vs-rest on 10-class MNIST). The math: for binary labels with positive rate `q`, the constant prediction `p = q` gives MSE = `q(1-q)` ≈ 0.09 — easy to find and hard for random trees to beat.

Two fixes applied above:

1. **Class-balanced sampling** (1:1 positive:negative): pushes the constant-MSE up to 0.25, which any real feature should beat. Eliminates the local minimum.
2. **Require `image` in `used_features(tree)`**: explicit constraint rejecting pure-constant and bare-variable trees that don't actually look at the image.

## Run K=1 single-feature smoke test (digit 0 vs rest)

Validates the pipeline before committing to the full 10-feature run. Target time: ~2-5 minutes.

In [ ]:
print('Running single-feature SR (digit 0 vs rest)...')
t0 = time.time()
tree_0 = discover_feature_one_vs_rest(X_train, y_train, target_class=0,
                                        pop_size=60, n_gens=20,
                                        seed=2026, verbose=True)
print(f'Done in {time.time() - t0:.1f}s')
print()
print('Discovered tree:')
print(str(tree_0))

# Evaluate as a 0-vs-rest binary classifier
from sklearn.metrics import accuracy_score, roc_auc_score
preds_train = predict_batch(tree_0, X_train)
preds_test = predict_batch(tree_0, X_test)
y_train_bin = (y_train == 0).astype(int)
y_test_bin = (y_test == 0).astype(int)

# Use train-median as threshold (robust)
thr = np.nanmedian(preds_train)
bin_train = (preds_train > thr).astype(int)
bin_test = (preds_test > thr).astype(int)

# Polarity-flexible accuracy
acc_a_train = (bin_train == y_train_bin).mean()
acc_b_train = ((1-bin_train) == y_train_bin).mean()
polarity = 1 if acc_a_train > acc_b_train else -1

acc_train = max(acc_a_train, acc_b_train)
bin_test_final = bin_test if polarity == 1 else (1 - bin_test)
acc_test = (bin_test_final == y_test_bin).mean()

# AUC handles any monotonic transform of the predictions
valid_mask = np.isfinite(preds_test)
auc_test = roc_auc_score(y_test_bin[valid_mask], preds_test[valid_mask] * polarity)

print(f'\nSingle-feature 0-vs-rest:  train acc={acc_train:.3f}  test acc={acc_test:.3f}  test AUC={auc_test:.3f}')
print('(if test acc > 0.92, single-feature is doing well; expect K=10 ensemble to reach ~90%+)')

## Run K=10 features (one per digit) and build multi-class classifier

Target time: 30-60 minutes on Colab GPU.

In [ ]:
K = 10
trees = [None] * K
t0 = time.time()
for k in range(K):
    print(f'\n=== Feature {k+1}/{K} (digit {k} vs rest) ===')
    tk = time.time()
    trees[k] = discover_feature_one_vs_rest(
        X_train, y_train, target_class=k,
        pop_size=60, n_gens=20, seed=2026 + k, verbose=True
    )
    print(f'feature {k} runtime: {time.time()-tk:.1f}s')
    print(f'tree {k}: {str(trees[k])[:120]}')
print(f'\nTotal SR runtime: {(time.time()-t0)/60:.1f} min')

In [ ]:
# Build feature matrices
print('Computing feature matrices...')
F_train = np.column_stack([predict_batch(t, X_train) for t in trees])    # [N_train, K]
F_test = np.column_stack([predict_batch(t, X_test) for t in trees])      # [N_test, K]

# Replace any NaN/inf with 0 (or feature mean)
F_train = np.nan_to_num(F_train, nan=0.0, posinf=0.0, neginf=0.0)
F_test = np.nan_to_num(F_test, nan=0.0, posinf=0.0, neginf=0.0)

print(f'F_train shape: {F_train.shape}, F_test shape: {F_test.shape}')
print('Feature value ranges (train):')
for k in range(K):
    print(f'  feature {k}: min={F_train[:, k].min():.3f} max={F_train[:, k].max():.3f} std={F_train[:, k].std():.3f}')

In [ ]:
# Train linear classifier on the K features
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

clf = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=1000, C=1.0, solver='lbfgs')   # multinomial is default for multi-class with lbfgs (newer sklearn removed the explicit arg)
)
clf.fit(F_train, y_train)

train_acc = clf.score(F_train, y_train)
test_acc = clf.score(F_test, y_test)

print(f'\n=== RESULTS ===')
print(f'Train accuracy: {train_acc:.4f}')
print(f'Test accuracy:  {test_acc:.4f}')
print(f'Goal: 0.95     {"✓ HIT" if test_acc >= 0.95 else "× missed by " + f"{(0.95 - test_acc)*100:.1f} pts"}')

In [ ]:
# Per-class breakdown
from sklearn.metrics import classification_report, confusion_matrix

y_pred = clf.predict(F_test)
print(classification_report(y_test, y_pred))
print('Confusion matrix:')
print(confusion_matrix(y_test, y_pred))

## Interpretability check

Look at what each discovered feature is. These are the tessera unique-value pitch — interpretable formulas, not black-box weights.

In [ ]:
for k in range(K):
    print(f'Feature {k} (target = digit {k}):')
    print(f'  tree (cx={complexity(trees[k])}): {str(trees[k])}')
    print()

## Verdict and next steps

**If test accuracy ≥ 95%:** Goal 3 achieved with K=10 features. Notebook validates the symbolic-regression-as-CNN-substitute pitch.

**If test accuracy is 85-94%:** The pipeline works but the ceiling for K=10 features on this subset is below 95%. Try:
1. **More features**: bump K to 20-30 (different seeds per class), let LogisticRegression learn weights
2. **More training samples**: bump N_TRAIN to 60000 (full MNIST). Currently using 5000 to keep the per-image loop tractable.
3. **Larger search budget**: pop=200, n_gens=100 instead of pop=60, n_gens=20

**If test accuracy is < 85%:** The SR pipeline isn't finding strong features. Possible issues:
1. Tree mutation distribution rarely produces FunctionalOp2D candidates → check by counting candidate types
2. Mean-pooling aggregation is too lossy → try `reduce_max` or stat-of-region features
3. Need a richer 2D operator vocabulary (Sobel, Gabor-like measures, etc.)

**Performance optimization path (if SR loop is too slow):**
1. Vmap the per-image evaluation over the batch axis (currently a Python for loop)
2. Use the GP integration with `use_jax_population_eval=True` (currently the custom loop bypasses it)
3. Add jax.grad-based const-opt (replaces scipy Nelder-Mead)

**The interpretability claim:** even if accuracy is below 95%, look at the discovered features. If they're recognizable image operators (Laplacians, Sobels, region statistics), that validates the symbolic-CV pitch — the model is **readable**, which a CNN isn't.
